<a href="https://colab.research.google.com/github/guanyuq03/card-and-krueger-did-replication/blob/main/notebooks/02_Replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import statsmodels.formula.api as smf

df = pd.read_csv("https://raw.githubusercontent.com/guanyuq03/card-and-krueger-did-replication/refs/heads/main/data/processed/njmin_clean.csv")
df.head()

,SHEET,CHAIN,CO_OWNED,STATE,SOUTHJ,CENTRALJ,NORTHJ,PA1,PA2,SHORE,...,FIRSTIN2,SPECIAL2,MEALS2,OPEN2R,HRSOPEN2,PSODA2,PFRY2,PENTREE2,NREGS2,NREGS112
0,46,1,0,0,0,0,0,1,0,0,...,0.08,1.0,2.0,6.5,16.5,1.03,NaN,0.94,4.0,4.0
1,49,2,0,0,0,0,0,1,0,0,...,0.05,0.0,2.0,10.0,13.0,1.01,0.89,2.35,4.0,4.0
2,506,2,1,0,0,0,0,1,0,0,...,0.25,NaN,1.0,11.0,11.0,0.95,0.74,2.33,4.0,3.0
3,56,4,1,0,0,0,0,1,0,0,...,0.15,0.0,2.0,10.0,12.0,0.92,0.79,0.87,2.0,2.0
4,61,4,1,0,0,0,0,1,0,0,...,0.15,0.0,2.0,10.0,12.0,1.01,0.84,0.95,2.0,2.0


In [2]:
# Full-time equivalent employment
df["fte_before"] = df["EMPFT"] + df["NMGRS"] + 0.5 * df["EMPPT"]
df["fte_after"] = df["EMPFT2"] + df["NMGRS2"] + 0.5 * df["EMPPT2"]

# Full meal price
df["fullmeal_before"] = df["PSODA"] + df["PFRY"] + df["PENTREE"]
df["fullmeal_after"] = df["PSODA2"] + df["PFRY2"] + df["PENTREE2"]

# Treatment indicator
df["treat"] = df["STATE"]

df[["SHEET", "treat", "fte_before", "fte_after", "WAGE_ST", "WAGE_ST2"]].head()

,SHEET,treat,fte_before,fte_after,WAGE_ST,WAGE_ST2
0,46,0,40.50,24.0,NaN,4.30
1,49,0,13.75,11.5,NaN,4.45
2,506,0,8.50,10.5,NaN,5.00
3,56,0,34.00,20.0,5.0,5.25
4,61,0,24.00,35.5,5.5,4.75


In [3]:
# Create a label for grouping
df["state_name"] = df["treat"].map({1: "NJ", 0: "PA"})

# Compute descriptive statistics
before_stats = df.groupby("state_name")[["fte_before", "WAGE_ST", "fullmeal_before"]].agg(["mean", "std"])
after_stats = df.groupby("state_name")[["fte_after", "WAGE_ST2", "fullmeal_after"]].agg(["mean", "std"])

# Display descriptive statistics tables
print("Before (Wave 1)")
display(before_stats)

print("After (Wave 2)")
display(after_stats)

Before (Wave 1)


fte_before              WAGE_ST           fullmeal_before          
                 mean        std      mean       std            mean       std
state_name                                                                    
NJ          20.439408   9.106239  4.612134  0.346351        3.351061  0.644463
PA          23.331169  11.856283  4.630132  0.351687        3.042368  0.601090

After (Wave 2)


fte_after            WAGE_ST2           fullmeal_after          
                 mean       std      mean       std           mean       std
state_name                                                                  
NJ          21.027429  9.293024  5.080849  0.104545       3.414754  0.635643
PA          21.165584  8.276732  4.617465  0.357478       3.026620  0.566291

In [4]:
# Calculate mean employment for NJ and PA before the policy
mean_nj_before = df.loc[df["treat"] == 1, "fte_before"].mean()
mean_nj_after = df.loc[df["treat"] == 1, "fte_after"].mean()
# Calculate mean employment for NJ and PA after the policy
mean_pa_before = df.loc[df["treat"] == 0, "fte_before"].mean()
mean_pa_after = df.loc[df["treat"] == 0, "fte_after"].mean()
# Compute the DID estimate
did_manual = (mean_nj_after - mean_nj_before) - (mean_pa_after - mean_pa_before)
# Print results
print("NJ before:", mean_nj_before)
print("NJ after:", mean_nj_after)
print("PA before:", mean_pa_before)
print("PA after:", mean_pa_after)
print("Manual DID estimate:", did_manual)

NJ before: 20.439408099688475
NJ after: 21.02742946708464
PA before: 23.33116883116883
PA after: 21.165584415584416
Manual DID estimate: 2.753605782980582


In [5]:
# Data for Wave 1
before = df[["SHEET", "treat", "fte_before"]].copy()
before["post"] = 0
before = before.rename(columns={"fte_before": "employment"})
# Data for Wave 2
after = df[["SHEET", "treat", "fte_after"]].copy()
after["post"] = 1
after = after.rename(columns={"fte_after": "employment"})
# Combine the two waves into one dataset
df_long = pd.concat([before, after], ignore_index=True)

In [6]:
# Keep only variables used in regression and drop missing values
did_data = df_long[["SHEET", "employment", "treat", "post"]].dropna().copy()

# Check the dataset structure and make sure there are no missing values
print(did_data.head())
print(did_data.isna().sum())

# Run the DID regression
model = smf.ols("employment ~ treat * post", data=did_data)
results = model.fit(cov_type="cluster", cov_kwds={"groups": did_data["SHEET"]})

# Print regression results
print(results.summary())

   SHEET  employment  treat  post
0     46       40.50      0     0
1     49       13.75      0     0
2    506        8.50      0     0
3     56       34.00      0     0
4     61       24.00      0     0
SHEET         0
employment    0
treat         0
post          0
dtype: int64
                            OLS Regression Results                            
Dep. Variable:             employment   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     1.806
Date:                Sat, 07 Mar 2026   Prob (F-statistic):              0.146
Time:                        19:42:21   Log-Likelihood:                -2904.2
No. Observations:                 794   AIC:                             5816.
Df Residuals:                     790   BIC:                             5835.
Df Model:                           3                                         
Covarian